## Интерпретатор Python

https://www.youtube.com/watch?v=f-jFxO2Kzlo

Python — это интерпретируемый язык, а это значит, что он не требует компиляции, как C или C++. Вместо этого интерпретатор Python читает и выполняет исходный код напрямую.

Когда сценарий Python запускается в первый раз, интерпретатор проверяет синтаксис и компилирует его в байтовый код, представляющий собой низкоуровневое представление исходного кода (создаёт файлы `pycache`). Затем этот байт-код выполняется виртуальной машиной Python (PVM), которая представляет собой интерпретатор, который считывает байт-код и выполняет его.

In [ ]:
import dis

def func_(x):
    return x % 2

# Показать байт-код функции
dis.dis(func_)

Каталог `pycache` автоматически создается интерпретатором Python и используется для хранения скомпилированных файлов байт-кода. Каталог содержит скомпилированные версии импортированных сценариев Python, а также любые модули, импортированные этими сценариями. 

Файлы .pyc – это скомпилированные байт-коды Python, которые создаются при импорте модулей. Кроме того, файлы .pyc также могут использоваться для распространения скомпилированных версий модулей или приложений

Файлы .pyc являются специфичными для версии Python, так что файлы, созданные для одной версии Python, не будут работать с другой версией.

## Выделение памяти

Переменная в Python - это label, которая имеет только ссылку на объект. 

Объект хранит:
- служебный заголовок PyObject с двумя полями:
    - ob_refcnt — счётчик ссылок (кол-во переменны/структур данных, указывающих на него)
    - ob_type — указатель на тип (который также хранит поведение: методы, операции, протоколы)
- данные объекта

Области памяти: Stack и Heap

```python
def foo():
    x = [1, 2, 3]
    y = 10
    return x, y
```

В Stack упорядочено (LIFO) хранятся служебные данные вызова функции foo, ссылки (указатели) на Python-объекты для локальных имён x, y

В Heap хранятся объект списка [1,2,3], три объекта 1, 2, 3, объект 10, кортеж ( [1,2,3], 10 ), объект фрейма (описание foo)

В куче объекты хранятся произвольным образом, что может привести к большой фрагментации. Для этого созданы структуры, позволяющие дробить память и использовать её последовательно.

PyMalloc — это специализированный аллокатор CPython для малых объектов: размером ≤ 512 байт. Он сидит поверх системного malloc() (обычный кучевой аллокатор ОС) и использует собственные структуры:
- Арена — 256 KB. Memory Manager интерпретатора запрашивает память для новой арены у ОС через PyMalloc. Если нужно хранить объект > 512 байт, то запрос идёт напрямую к malloc(). Арена хранит 64 пула. Арены сортируются по количеству свободных пулов в обратном порядке (для оптимального переиспользования освобождаемой памяти)
- Пул — 4 KB внутри арены. Пул хранит фиксированное число блоков одинакового размера. Заполненные пулы - fullpools, заполненные частично пулы - usedpools, незаполненные пулы - freepools. 
- Блок — фиксированный кусок памяти под объект (8 - 512 байт). Незанятые блоки - UNTACHED, занятые - ALLOCATED, освобождённые блоки - FREE (данные уже не используются, но блоки не отданы ОС, имеют приоритет при записи новых данных).

![Python Memory Manager](img/memory_management_in_python.drawio.png)

Переалокация - повторное выделение участка памяти (обычно большего, реже меньшего) с копированием существующих данных и освобождением старого участка

In [ ]:
import sys

boolean = True
print(sys.getsizeof(boolean)) # вывод размера булевой переменной (28 байт)

num = 10
print(sys.getsizeof(num)) # вывод размера целого числа (28 байт)

big_num = 10000
print(sys.getsizeof(big_num)) # вывод размера большого целого числа (28 байт)

float_num = 5.3
print(sys.getsizeof(float_num)) # вывод размера числа с плавающей точкой (24 байта)

string = 'hello'
print(sys.getsizeof(string)) # вывод размера строки (54 байта)

big_string = 'Равным образом реализация намеченных плановых заданий обеспечивает широкому кругу (специалистов) участие в формировании соответствующий условий активизации.'
print(sys.getsizeof(big_string)) # вывод размера большой строки (386 байт)

list_a = []
for i in range(10):
    list_a.append(i)
    print(i, sys.getsizeof(list_a)) # вывод размера списка при добавлении элементов

dict_a = {}
for i, v in enumerate(range(10)):
    dict_a[i] = v
    print(i, sys.getsizeof(dict_a)) # вывод размера словаря при добавлении элементов

28
28
28
24
54
386
0 88
1 88
2 88
3 88
4 120
5 120
6 120
7 120
8 184
9 184
0 232
1 232
2 232
3 232
4 232
5 360
6 360
7 360
8 360
9 360


## Переалокация памяти

Это процесс автоматического расширения динамической структуры данных, когда ей становится недостаточно текущего выделенного пространства: система выделяет новый, более крупный блок памяти, копирует в него все существующие элементы и освобождает старый блок, после чего структура продолжает работу уже с увеличенной ёмкостью

In [ ]:
# Каждый вызов append() добавляет элемент в конец списка.
# Когда список заполняет текущий выделенный блок памяти, Python делает переаллокацию:
# выделяет новый блок большего размера, копирует элементы и освобождает старый.
# Поэтому при большом числе append() происходит много мелких переаллокаций.
squares = []
for x in range(10_000_000):
    squares.append(x * x)


# Python пытается заранее определить целевой размер списка (например, по длине range()).
# Поэтому список создаётся сразу с большим запасом памяти под будущие элементы.
# Добавление элементов происходит без вызова append(), напрямую на уровне C-цикла.
# Если заранее выделенной памяти всё же не хватает (редко), вызывается внутренняя функция list_resize(),
# которая увеличивает ёмкость списка примерно на 12.5% от текущего размера.
squares = [x * x for x in range(10_000_000)]

В CPython 3.10–3.12 оптмизировали и list.append() стал заметно быстрее, а также цикл for тоже ускорен. Механизм роста списка стал менее затратным

In [ ]:
import timeit

# Тест 1: append() в цикле
def test_append():
    squares = []
    for x in range(10_000_000):
        squares.append(x * x)

# Тест 2: list comprehension
def test_comprehension():
    squares = [x * x for x in range(10_000_000)]

# Запускаем замеры
append_time = timeit.timeit("test_append()", globals=globals(), number=5)
comp_time   = timeit.timeit("test_comprehension()", globals=globals(), number=5)

print(f"append():         {append_time:.3f} сек")
print(f"list comp:        {comp_time:.3f} сек")
print(f"Ускорение:        {append_time / comp_time:.2f}x")

append():         2.009 сек
list comp:        1.733 сек
Ускорение:        1.16x


## Устройство списка

Список хранит не сами объекты, а ссылки (указатели) на них в памяти (`PyObject*[]`)

### Получение элемента

- проверяет, что i в диапазоне
- вычисляет адрес элемента: `address = base_address + i * sizeof(PyObject*)`
- берёт указатель и увеличивает счётчик ссылок (`INCREF`)
- возвращает объект, на который ссылается указатель

### Добавление элемента `x` в конец списка

- проверяет, есть ли свободное место (сравнивает `len(lst)` и `allocated`), если нет - переалокация
- записывает указатель на объект `x` в ячейку `items[len]`
- увеличивается счётчик ссылок у `x` (INCREF)
- увеличивается `len`

### Добавление элемента `x` в выбранное место `i`

- проверяет корректность индекса `i` и хватает ли выделенной памяти (`allocated`)
- все элементы с индексами `i … len-1` сдвигаются на одну позицию вправо и копируются указатели (PyObject*)
- в позицию `i` записывается указатель на `x`
- увеличивается счётчик ссылок у `x` (INCREF)
- увеличивается `len`

### Переалокация

В списках используется over-allocation: если при добавлении элемента len(lst) == allocated, то Python вычисляет новое значение allocated (с запасом), выделяет новый непрерывный массив указателей, копирует старые элементы, освобождает старый массив

## Устройство словаря

Словарь - хеш-таблица, оптимизированная для быстрого доступа по ключу и сохранения порядка вставки (Python ≥ 3.7)

Состоит из двух частей:

1. Sparse array (таблица слотов): разрежённый массив фиксированного размера, где каждый слот содержит индекс на запись или состояние EMPTY / DELETED
2. Compact entries (таблица записей): плотный массив записей (hash, key, value), которые идут в порядке вставки

### Получение элемента

```python
d = {"a": 10, "b": 20, "c": 30}
value = d["b"]
```

1. Вычисление хеша ключа (64-битное число)
    `h = hash("b")`
2. Определение позиции в sparse array. Количество слотов sparse array всегда степень двойки, например 8, 16, 32
    `slot = h & (size - 1)`
3. Смотрим содержимое sparse array
    `index = sparse_array[slot]`
    - index >= 0 → это индекс элемента в compact entries
    - index == -1 → пустая ячейка → ключа нет в словаре
    - index == -2 → deleted entry → продолжаем поиск
4. Идём в compact entries
    `entry = entries[index]`
    Получаем запись вида: `(h, key, value)`
5. Проверка hash и key
    - сравниваем хэши:
    ```python
    if entry.hash != h:
        → коллизия → пробуем следующий slot
    ```
    - сравниваем ключи:
    ```python
    if entry.key is key:
        → успех
    если нет, вызывается == для ключей
    ```
6. Возвращаем значение
    `return entry.value`

### Переалокация

1. Если коэффициент загрузки sparse array стал > 2/3 (занятые слоты / общий размер), то создаётся новый sparse array (хэш-таблица индексов `slot, index`) большего размера, существующие active entries (key, value, hash) перевставляются (reinsert), таблица индексов перезаполняется заново

2. Если в sparse array накопились многие слоты типа "deleted", то Python переаллоцирует таблицу, удаляет мусор, формирует компактный layout

In [11]:
# Имплементация словаря с использованием хеш-таблицы
class MyDictionary:
    def __init__(self):
        """
        Создаём хеш-таблицу. table — это список "корзин". Каждая корзина — список пар (key, value).
        """
        self.size = 5  # маленький размер, чтобы увидеть коллизии
        self.table = [[] for _ in range(self.size)]

    def __setitem__(self, key, value):
        """ Сохраняем значение по ключу """
        index = hash(key) % self.size
        bucket = self.table[index]

        # Проверяем: есть ли такой ключ в корзине
        for i, (k, v) in enumerate(bucket):
            if k == key:
                # Если ключ найден — обновляем значение
                bucket[i] = (key, value)
                return

        # Если ключа нет — добавляем новую пару
        bucket.append((key, value))

    def __getitem__(self, key):
        """ Получаем значение по ключу """
        index = hash(key) % self.size
        bucket = self.table[index]

        # Ищем нужный ключ в корзине
        for k, v in bucket:
            if k == key:
                return v

        # Если ключ не найден — ошибка
        raise KeyError(key)

    def __repr__(self):
        """ Вывод таблицы для отладки """
        return '\n'.join(
            f"{i}: {bucket}"
            for i, bucket in enumerate(self.table)
        )

d = MyDictionary()

d['apple'] = 'red'
d['banana'] = 'yellow'
d['cherry'] = 'red'

print(d['apple'])    # red
print(d['banana'])   # yellow
print(d['cherry'])   # red

print("\nВнутреннее состояние таблицы:")
print(d)

red
yellow
red

Внутреннее состояние таблицы:
0: []
1: []
2: [('apple', 'red'), ('banana', 'yellow')]
3: [('cherry', 'red')]
4: []


## Операторы == и is
Оператор is в Python используется для сравнения идентичности объектов, то есть — проверяет, указывают ли две переменные на один и тот же объект в памяти

In [2]:
a = [1, 2, 3]
b = [1, 2, 3]
print(a == b)
print(a is b)

a = b
print(a == b)
print(a is b)

True
False
True
True


## Проверка типов

In [ ]:
print(type(5))         # <class 'int'>
print(type("hello"))   # <class 'str'>

# но при проверке не считает подкласс за такой же тип
x = 5
if type(x) is int:
    print(x)

# проверяет, является ли объект экземпляром класса или его наследником
print(isinstance(5, int))                 # True
print(isinstance(True, int))              # True (bool — подкласс int)
print(isinstance([1, 2], (list, tuple)))  # True для одного из

# проверяет элементы в списке
x = [1, 2, 3, 4, 5]
if all(isinstance(i, int) for i in x):
    print("Все элементы — числа")

# проверяет пользовательские типы
class A: pass
class B(A): pass

b = B()

print(isinstance(b, A))   # True (B наследует A)

# проверяет по методу "duck typing"
def f(x):
    if hasattr(x, "__len__"):
        print("У объекта есть длина:", len(x))

f([1, 2, 3])

<class 'int'>
<class 'str'>
5
True
True
True
Все элементы — числа
True
У объекта есть длина: 3


## Итернирование
Интернирование — это механизм оптимизации, при котором одинаковые неизменяемые объекты (например, строки и числа) хранятся в памяти один раз и переиспользуются

In [ ]:
# Python заранее хранит в памяти числа от -5 до 256, потому что они часто используются (в циклах, индексах и т.п.).
# Интерпретатор не создаёт два разных объекта. Он берёт один и тот же объект числа из внутреннего пула (кэша) целых чисел
num_a = 5
num_b = 5
print(num_a is num_b) # True

# Число не интернировано (>256)
big_num_a = 1000
big_num_b = 1000
print(big_num_a is big_num_b) # False

# Python тоже кэширует строки, но не все, а только «простые» — то есть литералы без пробелов и специальных символов (обычно до 20 символов)
a = "hello"
b = "hello"
print(a is b)  # True

# Cтрока не интернирована (есть пробел)
a = "hello world"
b = "hello world"
print(a is b)  # False

# Интернирование работает только для неизменяемых типов
list_a = [1, 2, 3, 4, 5]
list_b = [1, 2, 3, 4, 5]
print(list_a is list_b)  # False

# Интернирование вручную
import sys
a = sys.intern("hello world")
b = sys.intern("hello world")
print(a is b)  # True

True
False
True
False
False
True


## Присваивание объекта, поверхностная и глубокая копии

In [37]:
# Присваивание
a = [1, [2]]
b = a
b.append(3)
b[1].append(4)
# Объект a изменился, так как b ссылается на него
print(a)

# Поверхностное копирование
a = [1, [2]]
import copy
b = copy.copy(a)
b.append(3)
b[1].append(4)
# Изменился только вложенный список в объекте a
print(a)

# Глубокое копирование
a = [1, [2]]
import copy
b = copy.deepcopy(a)
b.append(3)
b[1].append(4)
# Объект a остался прежним
print(a)

# Варианты создания поверхностной копии:
b = list(a)
b = a[:]
b = [i for i in a]
b = a.copy()

[1, [2, 4], 3]
[1, [2, 4]]
[1, [2]]


## Работа операторов and, or, not

In [40]:
print(not {1, 2, 3})
print([] and 1)
print([] or 1)

False
[]
1


## Примеры Comprehensions

In [5]:
# list comprehensions
list_ = [ i for i in range(5) if i % 2 == 0 ]
print(list_)

# set comprehensions
set_ = { i for i in range(5) if i % 2 == 0 }
print(set_)

# dict comprehensions
dict_ = { i: i**2 for i in range(5) if i % 2 == 0 }
print(dict_)

[0, 2, 4]
{0, 2, 4}
{0: 0, 2: 4, 4: 16}


## Перебор элементов с помощью range, enumerate, zip

In [5]:
num = 3

# range позволяет итерироваться по последовательности чисел
for n in range(1, num, 2):
    print(f"Значение: {n}")

fruits = ["apple", "banana", "cherry"]

# enumerate позволяет итерироваться по элементам с индексами
for index, fruit in enumerate(fruits):
    print(index, fruit)

a = [1, 2, 3]
b = [4, 5, 6, 7]
c = [8, 9]

# zip позволяет итерироваться параллельно по нескольким итерируемым объектам
for i in zip(a, b, c):
    print(i)

Значение: 1
0 apple
1 banana
2 cherry
(1, 4, 8)
(2, 5, 9)


## GC

Когда объект в Python больше не нужен (например, после того как на него уже нет ссылок), он помечается как garbage (мусор), после чего он будет автоматически удален при следующем запуске сборщика мусора (Garbage collector)

Сборка мусора в Python использует алгоритм под названием reference counting, который подсчитывает количество ссылок на каждый объект в памяти. Когда количество ссылок на объект становится равным нулю, он помечается как мусор и память автоматически освобождается.

В Python также реализованы другие алгоритмы сборки мусора, такие как generational garbage collection, который разбивает объекты на несколько “поколений” и собирает мусор с различной частотой в зависимости от поколения, в котором они находятся, но reference counting является основой управления памятью в Python.

In [ ]:
# Получение количества ссылок на объект
import sys
arr_1 = []
arr_2 = arr_1
print(sys.getrefcount(arr_1))

In [ ]:
# Очистка неиспользуемых объектов
import gc
gc.collect()

Для отладки утечек памяти в Python можно использовать инструменты:
- Memory Profiler: запустите вашу программу с помощью команды python -m memory_profiler my_script.py. Будет выведен подробный отчет о том, сколько памяти используется в каждой строке программы, а также общее использование памяти и любые утечки.
- objgraph: создаст изображение my_list.png, на котором будут показаны все объекты, на которые ссылается my_list, а также все объекты, которые ссылается на них

In [ ]:
from memory_profiler import profile

@profile
def my_func():
    # Some code that may cause a memory leak
    pass

In [ ]:
import objgraph

my_list = [1, 2, 3]
objgraph.show_refs([my_list], filename='my_list.png')

## GIL

https://tproger.ru/translations/global-interpreter-lock-guide

Механизм в CPython, который гарантирует, что только один поток исполнения может выполнять байт-код Python в любой момент времени. Используется для обеспечения безопасности потоков в многопоточной среде и для упрощения реализации интерпретатора.

## Рекурсия

In [ ]:
def factorial(n):
    # Используется больше памяти, но компактнее
    # Имеется ограничение 1000 вызовов
    if n == 1:
        return n
    else:
        return n * factorial(n - 1)


def factorial(n):
    num = 1
    while n >= 1:
        num = num * n
        n = n - 1
    return num

## Обработка исключений

In [3]:
a = 10
b = 0

try:
    # выполняется всегда, сюда помещаем код, который может вылетать с ошибкой
    result = a / b   # может выбросить ZeroDivisionError
except ZeroDivisionError:
    # выполняется в том случае, если try поймал указанные ошибки (можно использовать несколько except)
    result = None  # резервное значение
else:
    # выполняется в том случае, если try не поймал ошибок
    ...
finally:
    # выполняется в любом случае
    ...

print(result)

None


## Указание нескольких except

In [135]:
try:
    raise ValueError(1)
except ValueError:
    # Отработает только первый блок
    print("ValueError")
except Exception:
    # Если этот except будет первее, то он обработается раньше, так как это базовый класс исключений
    print("Exception")

ValueError


In [137]:
try:
    raise TypeError(1)
except (ValueError, TypeError):
    # Если исключение подойдёт под один из указанных типов, то оно будет обработано
    print("Value or Type Error")
except SyntaxError:
    print("SyntaxError")

# Если исключение не подошло под указанные в Except, то его обработает интерпретатор

Value or Type Error


In [139]:
try:
    raise GeneratorExit
except Exception:
    print("Exception")
except:
    # Ловит базовый класс всех Exception (BaseException)
    print("Base Exception")

Base Exception


## Переподнятие Expection

In [142]:
try:
    try:
        raise ValueError(1)
    except Exception:
        # Если внутренний обработчик поймал ошибку, то внешний обработчик её не обнаружит
        print("INNER")
        # Если добавить raise, то её обработает и внешний обработчик
        raise
except Exception:
    print("OUTER")

INNER
OUTER


## Группы Exception

In [1]:
!python --version
# появилось в Python 3.11: ExceptionGroup и BaseExceptionGroup
# Позволяет обрабатывать группы исключений параллельно (например, из asyncio)

try:
    eg = ExceptionGroup(
        "exception group",
        [
            TypeError("Некорректный тип"),
            ValueError("Неверное значение"),
        ]
    )
    raise eg

# Блок except перехватывает всю группу целиком
except ExceptionGroup as e:
    print("Полностью пойманная группа исключений:")
    print(e)


try:
    eg = ExceptionGroup(
        "exception group",
        [
            TypeError("Некорректный тип"),
            ValueError("Неверное значение"),
        ]
    )
    raise eg

# Обработка отдельных типов исключений — синтаксис except*
except* TypeError as e:
    print("\nОбработка TypeError:")
    print(e)

except* ValueError as e:
    print("\nОбработка ValueError:")
    print(e)

Python 3.12.11
Полностью пойманная группа исключений:
exception group (2 sub-exceptions)

Обработка TypeError:
exception group (1 sub-exception)

Обработка ValueError:
exception group (1 sub-exception)


## Свой тип исключения

In [4]:
# Получение более информативного вывода о конкретных ошибках
class Custom(Exception):
    def __init__(self, value, message = "Default message"):
        self.value = value
        self.message = message
        super().__init__(self.message)

raise Custom(500, message="My Error")

Custom: My Error

## Match-Case

In [10]:
def process_value(value):
    match value:
        case 1:
            print("Value is 1")
        case 2 | 3 | 4:
            print("Value is 2 or 3 or 4")
        case _:
            print("Value is not 1 or 2 or 3 or 4")

process_value(1) # output: Value is 1
process_value(3) # output: Value is not 1 or 2

Value is 1
Value is 2 or 3 or 4
